In [20]:
import conllu
from collections import defaultdict
import pandas as pd

In [ ]:
sequences = []

with open('en_ewt-ud-train.conllu', mode='rt', encoding='utf-8') as f:
  data = conllu.parse(f.read())

for datum in data:
  tags = [token["upos"]for token in datum] 
  sequences.append(tags)

print(f"Loaded {len(sequences):,} sentences.")

In [17]:
BOS = "<BOS>"
EOS = "<EOS>"

counts: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))

for seq in sequences:
    padded = [BOS] + seq + [EOS]
    for cur, nxt in zip(padded, padded[1:]):
        counts[cur][nxt] += 1

In [ ]:
def make_transition_matrix(counts: dict, add_k: float = 0.0):
    tags = sorted(counts.keys())
    probs = {}
    for cur in tags:
        total = sum(counts[cur].values()) + add_k * len(tags)
        probs[cur] = {
            nxt: (counts[cur][nxt] + add_k) / total
            for nxt in tags
        }
    return probs

probs = make_transition_matrix(counts, add_k=0.0)

In [25]:
def top_k(current_tag: str, k: int = 5) -> list[tuple[str, float]]:
    row = probs.get(current_tag, {})
    return sorted(row.items(), key=lambda x: x[1], reverse=True)[:k]

def bot_k(current_tag: str, k: int = 5) -> list[tuple[str, float]]:
    row = probs.get(current_tag, {})
    return sorted(row.items(), key=lambda x: x[1], reverse=False)[:k]

def transition_prob(current_tag: str, next_tag: str) -> float:
    return probs.get(current_tag, {}).get(next_tag, 0.0)

In [22]:
all_tags = sorted({t for row in probs.values() for t in row} | set(probs.keys()))

matrix = pd.DataFrame(
    {cur: {nxt: probs.get(cur, {}).get(nxt, 0.0) for nxt in all_tags}
     for cur in all_tags},
    index=all_tags,
    columns=all_tags,
).T

pd.set_option("display.float_format", "{:.4f}".format)
print("\nTransition matrix  P(col | row):")
print(matrix.to_string())


Transition matrix  P(col | row):
       <BOS>    ADJ    ADP    ADV    AUX  CCONJ    DET   INTJ   NOUN    NUM   PART   PRON  PROPN  PUNCT  SCONJ    SYM   VERB      X      _
<BOS> 0.0000 0.0410 0.0432 0.0752 0.0254 0.0231 0.1004 0.0323 0.0615 0.0391 0.0047 0.2523 0.1244 0.0352 0.0356 0.0075 0.0596 0.0001 0.0392
ADJ   0.0000 0.0544 0.0792 0.0139 0.0033 0.0428 0.0051 0.0004 0.5163 0.0083 0.0308 0.0110 0.0649 0.1279 0.0232 0.0015 0.0084 0.0011 0.0038
ADP   0.0000 0.0715 0.0279 0.0160 0.0008 0.0059 0.3599 0.0003 0.1625 0.0393 0.0008 0.1337 0.1342 0.0243 0.0028 0.0033 0.0064 0.0007 0.0095
ADV   0.0000 0.1402 0.0914 0.0885 0.0408 0.0258 0.0453 0.0010 0.0142 0.0179 0.0185 0.0839 0.0094 0.1715 0.0354 0.0031 0.1941 0.0014 0.0143
AUX   0.0000 0.1070 0.0323 0.1404 0.0790 0.0020 0.0804 0.0003 0.0118 0.0090 0.1186 0.0467 0.0066 0.0164 0.0083 0.0009 0.3384 0.0001 0.0017
CCONJ 0.0000 0.0878 0.0277 0.0821 0.0489 0.0001 0.0972 0.0030 0.1449 0.0144 0.0157 0.1675 0.0764 0.0130 0.0170 0.0040 0.1733 0.0001 

In [27]:
for tag in ["DET", "NOUN", "VERB", "ADJ", BOS]:
    top = top_k(tag, k=5)
    bot = bot_k(tag, k=5)
    
    print(f"\nMost likely tags after {tag!r}:")
    for nxt, p in top:
        print(f"  {nxt:<8} {p:.4f}")
    print(f"\nLeast likely tags after {tag!r}:")
    for nxt, p in bot:
        print(f"  {nxt:<8} {p:.4f}")
  


Most likely tags after 'DET':
  NOUN     0.5876
  ADJ      0.2339
  PROPN    0.0758
  VERB     0.0190
  ADV      0.0158

Least likely tags after 'DET':
  <BOS>    0.0000
  INTJ     0.0000
  SCONJ    0.0001
  PART     0.0004
  CCONJ    0.0006

Most likely tags after 'NOUN':
  PUNCT    0.2894
  ADP      0.2060
  NOUN     0.1236
  AUX      0.0722
  CCONJ    0.0710

Least likely tags after 'NOUN':
  <BOS>    0.0000
  X        0.0005
  INTJ     0.0006
  SYM      0.0036
  _        0.0046

Most likely tags after 'VERB':
  DET      0.1845
  ADP      0.1845
  PRON     0.1693
  PUNCT    0.0789
  NOUN     0.0783

Least likely tags after 'VERB':
  <BOS>    0.0000
  X        0.0003
  INTJ     0.0010
  SYM      0.0019
  _        0.0060

Most likely tags after 'ADJ':
  NOUN     0.5163
  PUNCT    0.1279
  ADP      0.0792
  PROPN    0.0649
  ADJ      0.0544

Least likely tags after 'ADJ':
  <BOS>    0.0000
  INTJ     0.0004
  X        0.0011
  SYM      0.0015
  AUX      0.0033

Most likely tags after 